# The master tables — decodability BEFORE editability

Reads the `scores.json` files `master_eval.ipynb` writes (scanning `runs/**`, excluding
`runs/archive/` and `_`-prefixed topics) and renders them as **heatmap figures** —
publication-ready, copy-pasteable, and displayed inline. Nothing is cached to disk:
each run's `scores.json` is the single source of truth and cell [1] re-derives these in
under a second.

**Order is deliberate: decodability first.** An editability number is only interpretable
once the probes demonstrably read the state — an editor writing through a probe that
decodes nothing produces noise, not a negative result.

* **Table 1** — decodability (Probe Skill, the cross-environment axis: 1 = perfect,
  0 = trivial baseline; identical to R² on regression). Every probe is held out **by
  sequence** in both environments.
* **Table 1b** — discworld decodability **by component**. The aggregate is
  variance-weighted ~1000:1 toward position, so velocity is only visible here.
* **Table 2** — editability. Panel (a) is the Edit Index with each run's **unedited**
  floor as its first column — that is where the −1 end actually sits, so every editor
  must be read against it. Panel (b) is the guard, `RMSE(edited, GT)/RMSE(unsteered, GT)`
  at the edit step: **> 1 means the edit degraded the model rather than steering it**,
  whatever the index says.

One row per (run, basis): a basis is a different probe target, never a new column.

In [ ]:
# [1] Collect every scores.json (runs/**; archive/ and _topics excluded) -> tidy frames.
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.colors import TwoSlopeNorm

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
EDITORS = ("PI", "ND", "ND-sub", "GS")
COMPONENTS = ("o1·x", "o1·y", "o2·x", "o2·y", "o1·vx", "o1·vy", "o2·vx", "o2·vy")

rows, perdim_rows = [], []
for sp in sorted((REPO / "runs").rglob("scores.json")):
    rel = sp.relative_to(REPO / "runs")
    if rel.parts[0] == "archive" or rel.parts[0].startswith("_"):
        continue
    s = json.loads(sp.read_text())
    # A basis is a different probe TARGET, so it is a different ROW — never a column
    # that exists for only some runs.
    basis = s.get("basis", "cartesian" if s["env"] == "discworld" else "mine/theirs")
    row = {"topic": rel.parts[0], "run": sp.parent.name, "env": s["env"],
           "basis": basis, "arch": s["arch"], "val": s["val_loss"]}
    if s["env"] == "discworld":
        T = s["targets"]["pos"]
        F = s["targets"].get("full")
        row["skill_LIN"] = max(T["probe_skill_linear"])
        row["skill_MLP"] = max(T["probe_skill_mlp"])
        row["tripwire"] = (T["probe_sanity"]["n_violations"]
                           + (F["probe_sanity"]["n_violations"] if F else 0))
        row["unedited"] = T["unedited"]["edit_index"]
        best, key = T["best"], "edit_index"
        if F:   # per-component skill comes from the FULL target (velocity lives there)
            bp = int(np.argmax(F["probe_skill_linear"]))
            for name, pdim in (("LIN", F["probe_perdim_linear"][bp]),
                               ("MLP", F["probe_perdim_mlp"][bp])):
                perdim_rows.append({"run": row["run"], "topic": row["topic"],
                                    "basis": basis, "probe": name, "point": bp,
                                    **dict(zip(COMPONENTS, pdim[:len(COMPONENTS)]))})
    else:
        sk = s["probe_skill"]
        row["skill_LIN"] = max(sk.get("mine|linear|sequence", [np.nan]))
        row["skill_MLP"] = max(sk.get("mine|mlp|sequence", [np.nan]))
        row["tripwire"] = 0
        row["unedited"] = s["unedited"]["edit_index_union"]
        row["legal_mass"] = s["gates"]["legal_mass"]
        row["ce_excess"] = s["gates"]["ce"] - s["gates"]["bayes_ce"]
        best, key = s["best"], "edit_index_union"
    for ed in EDITORS:
        b = best.get(ed)
        row[f"{ed} EI"] = b[key] if b else np.nan
        row[f"{ed} fid"] = b["fidelity_ratio"] if b else np.nan
        row[f"{ed} arm"] = f"pt{b['point']} α{b['alpha']:g}" if b else "—"
    rows.append(row)

DF = pd.DataFrame(rows).sort_values(["topic", "env", "run", "basis"])
PERDIM = pd.DataFrame(perdim_rows)
LABEL = [f"{r.env} · {r.run}" + (f" · {r.basis}" if r.env == "discworld" else "")
         for r in DF.itertuples()]
print(f"{len(DF)} scored runs · {DF.env.value_counts().to_dict()}")

sns.set_theme(style="white", font_scale=0.95)
def heat(ax, data, xt, yt, *, fmt, cmap, norm=None, vmin=None, vmax=None, cbar_label,
         title):
    """One heatmap cell block — every table here is built through this, so the tables
    cannot drift apart in colormap, annotation format, or scaling."""
    sns.heatmap(data, annot=True, fmt=fmt, cmap=cmap, norm=norm, vmin=vmin, vmax=vmax,
                xticklabels=xt, yticklabels=yt, linewidths=1.2, linecolor="white",
                cbar_kws=dict(label=cbar_label, shrink=0.75, pad=0.02), ax=ax,
                annot_kws=dict(fontsize=9))
    ax.set_title(title, fontsize=10.5, loc="left", pad=8)
    ax.tick_params(labelsize=9, length=0)
    plt.setp(ax.get_yticklabels(), rotation=0)
    plt.setp(ax.get_xticklabels(), rotation=0)

In [ ]:
# [2] TABLE 1 — DECODABILITY. Read before Table 2 means anything.
#     Probe Skill at the best residual point; 1 = perfect, 0 = the trivial baseline.
#     All probes held out BY SEQUENCE in both environments.
fig, ax = plt.subplots(figsize=(4.4, 0.52 * len(DF) + 1.7))
heat(ax, DF[["skill_LIN", "skill_MLP"]].values, ["LIN", "MLP-128"], LABEL,
     fmt="+.3f", cmap="Greens", vmin=0.0, vmax=1.0,
     cbar_label="Probe Skill", title="Table 1 — decodability (best residual point)")
for i, t in enumerate(DF["tripwire"].values):      # tripwire = MLP < linear somewhere
    if t:
        ax.add_patch(plt.Rectangle((0, i), 2, 1, fill=False, edgecolor="#c0392b", lw=3))
        ax.text(2.06, i + 0.5, f"⚠ {t}", color="#c0392b", va="center", fontsize=9)
fig.suptitle("discworld = pos target · Othello = mine/theirs tiles;  a red box marks "
             "MLP < linear (decodability untrusted)", fontsize=8.5, y=0.02, color="0.35")
plt.show()

In [ ]:
# [3] TABLE 1b — DISCWORLD DECODABILITY BY COMPONENT (full target, best point).
#     The aggregate in Table 1 is variance-weighted ~1000:1 toward position; velocity
#     is only visible here. (Othello's per-tile equivalent is 64 columns — it stays in
#     each run's scores.json.)
if len(PERDIM):
    lab = [f"{r.run} · {r.basis} · {r.probe}" for r in PERDIM.itertuples()]
    fig, ax = plt.subplots(figsize=(8.6, 0.52 * len(PERDIM) + 1.7))
    heat(ax, PERDIM[list(COMPONENTS)].values, list(COMPONENTS), lab,
         fmt="+.3f", cmap="Greens", vmin=0.0, vmax=1.0, cbar_label="Probe Skill",
         title="Table 1b — discworld decodability by component")
    ax.axvline(4, color="#172239", lw=2)          # position | velocity
    n = len(PERDIM)
    ax.text(2, n + 0.62, "position", ha="center", fontsize=9.5, color="0.3")
    ax.text(6, n + 0.62, "velocity", ha="center", fontsize=9.5, color="0.3")
    plt.show()
else:
    print("no discworld runs with per-component decodability yet")

In [ ]:
# [4] TABLE 2 — EDITABILITY. Two panels, never mixed in one colour scale:
#     (a) Edit Index, prefixed by each run's UNEDITED floor — where its −1 end sits.
#     (b) the guard: RMSE(edited, GT)/RMSE(unsteered, GT) at the edit step.
#         >1 = degraded, not steered. Colormap REVERSED so green = good in both panels.
ei = DF[["unedited"] + [f"{e} EI" for e in EDITORS]].values
fid = DF[[f"{e} fid" for e in EDITORS]].values
fig, axes = plt.subplots(1, 2, figsize=(12.4, 0.52 * len(DF) + 2.0),
                         gridspec_kw=dict(width_ratios=[5, 4], wspace=0.35))
heat(axes[0], ei, ["unedited"] + list(EDITORS), LABEL, fmt="+.3f", cmap="RdYlGn",
     vmin=-1.0, vmax=1.0, cbar_label="Edit Index",
     title="(a) did the edit land?   +1 = the edited world, −1 = the unedited one")
axes[0].axvline(1, color="#172239", lw=2)          # floor | editors
heat(axes[1], fid, list(EDITORS), [""] * len(DF), fmt=".2f", cmap="RdYlGn_r",
     norm=TwoSlopeNorm(vmin=0.0, vcenter=1.0, vmax=3.0), cbar_label="fidelity ratio",
     title="(b) did the world survive?   1.0 = doing nothing")
fig.suptitle("Table 2 — editability, every canonical editor", fontsize=12, y=1.0)
plt.show()

# the winning arm behind each cell, so a number can always be traced to a configuration
print("best arm per cell (point, α):")
display(DF.set_index([ "env", "run"])[[f"{e} arm" for e in EDITORS]])